# Тема 26. Факторы мошенничества в карточных платежах и кредитовых переводах
**Страны ЕС с доступными наблюдениями · 2022–2024**

источник: https://data.ecb.europa.eu/data/datasets

## 2.1. Постановка задачи

**Бизнес-проблема.** Мошеннические карточные платежи и кредитовые переводы создают потери для банков и их клиентов. Сравнение сопоставимых сегментов помогает выбирать, куда направлять ресурсы на профилактику и мониторинг.

**Бизнес-цель.** Определить, какие характеристики операций и внешние факторы связаны с различиями в доле мошенничества. В прогнозной части оценить, дают ли эти факторы прирост точности относительно базового подхода.

**Объект** — карточные платежи и кредитовые переводы в странах ЕС. **Предмет** — методы объединения статистики ECB, опросных показателей Eurostat и регуляторных публикаций для анализа факторов мошенничества. **Границы** — страны ЕС с доступными данными, шесть полугодий с 2022-H1 по 2024-H2, карточные операции CP0, CW1, CP1, CP11 и кредитовые переводы PCT. В анализе сохраняются канал, дистанционность, функция карты и SCA при наличии данных.

**Исследовательский вопрос.** Как тип операции, канал, дистанционность её инициирования и распространённость онлайн-покупок связаны с различиями в доле мошенничества?

| Критерий достижения цели | Как проверяется |
|---|---|
| Рассчитать долю мошенничества по странам, полугодиям и типам операций | Сопоставить ряды ECB по всем существенным измерениям. Рассчитать долю по количеству и отдельно по сумме. |
| Оценить статистическую связь с внешними факторами | На этапе анализа учитывать повторные наблюдения и проверять, позволяет ли число независимых периодов оценить значимость. |
| Сравнить прогнозные модели | Использовать более ранние периоды для обучения и поздние для оценки. Сравнить с базовым прогнозом по MAE и RMSE. |
| Оценить важность признаков для бизнеса | Сопоставить ошибки при включении и исключении групп признаков на временной проверке. |

**H1.** Характеристики операций и внешние факторы связаны с `fraud_rate`. Добавление этих признаков снижает прогнозную ошибку. **H0.** Изучаемые признаки не связаны с `fraud_rate`, а их добавление не снижает прогнозную ошибку. Связь и прирост точности проверяются раздельно.

**Практическая ценность.** Сравнение сегментов помогает приоритизировать мониторинг, в том числе оценить, где стоит отдельно изучить применение усиленной аутентификации клиента (SCA). Подготовка данных в этом ноутбуке создаёт основу для проверки гипотез и обучения моделей. 

## 2.2. Подбор и изучение источников данных

| Источник | Формат и единица наблюдения | Роль в проверке гипотезы |
|---|---|---|
| [ECB PCP](https://data.ecb.europa.eu/data/datasets/PCP/data-information) | SDMX CSV. Страна × период × серия карточных операций. | Задаёт числитель и знаменатель доли по карточным платежам. Канал и дистанционность позволяют выделять сегменты. |
| [ECB PCT](https://data.ecb.europa.eu/data/datasets/PCT/data-information) | SDMX XML. Страна × период × серия кредитовых переводов. | Позволяет рассчитать сопоставимую долю для переводов и сравнить два вида операций. |
| [Eurostat isoc_ec_ib20](https://ec.europa.eu/eurostat/databrowser/product/view/isoc_ec_ib20?lang=en) | JSON-stat. Страна × год × индикатор × группа населения × единица. | После выбора одного показателя присоединяется по стране и году. Описывает распространённость онлайн-покупок. |
| EBA и ECB | Четыре официальные HTML-публикации. Одна строка на публикацию. | Годовые частоты терминов характеризуют темы выбранных публикаций и присоединяются по году. |

### Общие правила загрузки и проверки

`compact_profile` применяет единые проверки ко всем четырём источникам. Диапазон ECB упорядочен хронологически с учётом частоты. Пустые значения учитываются отдельно от нулей. Исходные файлы сохраняются в исходном виде.

In [2]:
# Задаём папки, период и страны, затем определяем общие функции проверки.
from pathlib import Path
import json
import os
import re
import xml.etree.ElementTree as ET
from itertools import product

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

ROOT = Path.cwd()
DATA_DIR = Path.home() / "Downloads" / "inputs"
CORE_COUNTRIES = ("FR", "IT", "ES")
EU_COUNTRIES = (
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "GR", "HU",
    "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
)
USE_EU_SCOPE = True
COUNTRIES = EU_COUNTRIES if USE_EU_SCOPE else CORE_COUNTRIES
YEARS = (2022, 2023, 2024)
HALVES = tuple(f"{year}-H{half}" for year in YEARS for half in (1, 2))

def local_file(name):
    options = [DATA_DIR / name, ROOT / name]
    for path in options:
        if path.is_file():
            return path
    raise FileNotFoundError(f"Поместите {name} в {DATA_DIR} или {ROOT}.")

def time_bounds(values):
    if pd.api.types.is_datetime64_any_dtype(values):
        return values.min(), values.max()
    labels = values.dropna().astype(str).unique()
    def sort_key(label):
        match = re.fullmatch(r"(\d{4})(?:-([SHQM]?)(\d{1,2}))?", label)
        if match is None:
            return (0, 0, label)
        year, freq, number = match.groups()
        width = {"H": 6, "S": 6, "Q": 3}.get(freq, 1)
        return (int(year), (int(number or 1) - 1) * width, label)
    ordered = sorted(labels, key=sort_key)
    return (ordered[0], ordered[-1]) if ordered else (None, None)

def compact_profile(df, name, date_col=None, key_cols=None, show=None):
    print(f"{name} · shape = {df.shape}")
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100).round(2),
    }))
    if date_col is not None:
        print("Диапазон:", *time_bounds(df[date_col]), sep=" ")
    if key_cols is not None:
        print("Дубликаты по", " + ".join(key_cols), "=", df.duplicated(key_cols).sum())
    display((df if show is None else df[show]).head(3))
    numeric = df.select_dtypes(include="number")
    if len(numeric.columns):
        display(numeric.describe().T)
    # Частоты выводятся только для полей, по которым проверяем отбор.
    top_fields = ("FRD_TYP", "UNIT_MEASURE", "FREQ", "OBS_STATUS", "TYP_TRNSCTN",
                  "RMT_INTTN", "INTTN_CHNNL")
    frequencies = []
    for column in (*top_fields, "KEY", "TITLE"):
        if column not in df:
            continue
        values = df[column]
        formatted = ""
        if column in top_fields:
            top = values.value_counts(dropna=False).head(10)
            formatted = " | ".join(
                f"{('<NA>' if pd.isna(value) else str(value))} ({count:,})"
                for value, count in top.items()
            )
        frequencies.append((column, values.nunique(), formatted))
    if frequencies:
        with pd.option_context("display.max_colwidth", 200):
            display(pd.DataFrame(frequencies, columns=["field", "nunique", "top_10"]))


PCP_DIMS = ["FREQ", "REF_AREA", "COUNT_AREA", "TRMNL_LCTN", "TYP_TRNSCTN",
            "RL_TRNSCTN", "INTTN_CHNNL", "RMT_INTTN", "PYMNT_SCHM",
            "CRD_FNCTN", "SCA", "FRD_TYP", "TRANSFORMATION", "UNIT_MEASURE"]
PCT_DIMS = ["FREQ", "REF_AREA", "COUNT_AREA", "RL_TRNSCTN", "INTTN_CHNNL",
            "RMT_INTTN", "PYMNT_SCHM", "SCA", "FRD_TYP", "TRANSFORMATION", "UNIT_MEASURE"]
ECB_NUMERIC = ["OBS_VALUE", "UNIT_MULT", "DECIMALS", "PRE_BREAK_VALUE"]
OBS_STATUS_LABELS = {
    "A": "Обычное значение", "P": "Предварительное", "E": "Оценочное",
    "Q": "Значение скрыто", "M": "Значение отсутствует",
    "L": "Данные существуют, но не были собраны",
}

def type_ecb(df, dims):
    required = set(dims + ["TIME_PERIOD", "OBS_VALUE", "UNIT_MULT"])
    if missing := required.difference(df.columns):
        raise ValueError(f"В выгрузке ECB отсутствуют поля: {sorted(missing)}")
    for col in set(ECB_NUMERIC).intersection(df.columns):
        df[col] = pd.to_numeric(df[col].replace({"NaN": np.nan, "": np.nan}), errors="raise")
    for col in dims + ["TIME_PERIOD"]:
        df[col] = df[col].astype("string")
    return df

def read_ecb_csv(path, dims):
    code_types = dict.fromkeys(dims + ["KEY", "TIME_PERIOD"], "string")
    return type_ecb(pd.read_csv(path, dtype=code_types, low_memory=False), dims)

def show_ecb_codes(frame):
    # Расшифровываем статусы отдельно, чтобы не повторять таблицу частот.
    if "OBS_STATUS" in frame:
        codes = frame["OBS_STATUS"].dropna().unique()
        display(pd.Series({code: OBS_STATUS_LABELS.get(code) for code in codes}, name="meaning"))
    value = frame["OBS_VALUE"]
    print(f"OBS_VALUE: пропусков {value.isna().sum():,}, нулей {value.eq(0).sum():,}, "
          f"отрицательных {value.lt(0).sum():,}.")
    if "TRANSFORMATION" in frame:
        display(frame.loc[value.lt(0), "TRANSFORMATION"].value_counts().rename("negative_rows"))

### 2.2.1. ECB PCP — карточные платежи и снятие наличных по картам (CSV)

Набор Card payments and cash withdrawals using cards (PCP) опубликован Европейским центральным банком на ECB Data Portal. Он содержит статистику карточных платежей и снятия наличных, в том числе сведения о мошеннических операциях. Одна строка соответствует одному наблюдению статистического ряда за один период. Ряд определяется комбинацией признаков, среди которых территория, тип операции, канал инициирования и единица измерения.

- REF_AREA — в какой стране (например, DE — Германия).

- TIME_PERIOD — за какой период (например, 2022-S1 — первое полугодие 2022).

- TYP_TRNSCTN — какой тип операции (CP0 — карточный платёж, CW1 — снятие наличных).

- INTTN_CHNNL — каким каналом (_T — все каналы, 2000 — электронно, 2223 — contactless).

- RMT_INTTN — дистанционно или нет (R — да, NR — нет, _T — все).

- UNIT_MEASURE — в чём измеряем (PN — количество штук, EUR — сумма в евро).

- FRD_TYP — это все операции или только мошеннические (_Z — все, F — мошеннические).

- OBS_VALUE — само число.

- TRANSFORMATION — уровень или темп роста.


`FRD_TYP=_Z` — весь поток в заданном срезе, `F` — мошеннические операции, а `F10` и другие коды — подтипы. Первичный профиль охватывает всю исходную выгрузку, включая годовые и квартальные ряды. Статусы интерпретируются по [перечню SDMX](https://circabc.europa.eu/sd/a/5a3385ad-6bf5-4fe9-a5af-e4f30d3ae01f/DOC_SP-2014-08-Annex%203_SDMX_code_lists_for_ESSPROS.pdf).

In [3]:
pcp_raw = read_ecb_csv(local_file("pcp.csv"), PCP_DIMS)
compact_profile(pcp_raw, "ECB PCP", "TIME_PERIOD", ["KEY", "TIME_PERIOD"])
show_ecb_codes(pcp_raw)

ECB PCP · shape = (1071509, 35)


,dtype,missing,missing_pct
KEY,string,0,0.00
FREQ,string,0,0.00
REF_AREA,string,0,0.00
COUNT_AREA,string,0,0.00
TRMNL_LCTN,string,0,0.00
TYP_TRNSCTN,string,0,0.00
RL_TRNSCTN,string,0,0.00
INTTN_CHNNL,string,0,0.00
RMT_INTTN,string,0,0.00
PYMNT_SCHM,string,0,0.00


Диапазон: 2000 2026-Q2
Дубликаты по KEY + TIME_PERIOD = 0


,KEY,FREQ,REF_AREA,COUNT_AREA,TRMNL_LCTN,TYP_TRNSCTN,RL_TRNSCTN,INTTN_CHNNL,RMT_INTTN,PYMNT_SCHM,...,DISS_ORG,TIME_PER_COLLECT,COVERAGE,DATA_COMP,DECIMALS,METHOD_REF,TITLE,TITLE_COMPL,UNIT,UNIT_MULT
0,PCP.A.AT.LT.W0.CP0.2._T._T.PCS_ALL._Z._X._Z.N.PN,A,AT,LT,W0,CP0,2,_T,_T,PCS_ALL,...,NaN,S,NaN,NaN,3,NaN,"Number of card payments, received from Lithuania","Number of card payments, received from Lithuania",PN,6
1,PCP.A.AT.LT.W0.CP0.2._T._T.PCS_ALL._Z._X._Z.N.PN,A,AT,LT,W0,CP0,2,_T,_T,PCS_ALL,...,NaN,S,NaN,NaN,3,NaN,"Number of card payments, received from Lithuania","Number of card payments, received from Lithuania",PN,6
2,PCP.A.AT.LT.W0.CP0.2._T._T.PCS_ALL._Z._X._Z.N.PN,A,AT,LT,W0,CP0,2,_T,_T,PCS_ALL,...,NaN,S,NaN,NaN,3,NaN,"Number of card payments, received from Lithuania","Number of card payments, received from Lithuania",PN,6


,count,mean,std,min,25%,50%,75%,max
OBS_VALUE,967573.0,10957.455301,187426.911221,-100.0,0.053219,6.291011,150.905259,2.326652e+07
PRE_BREAK_VALUE,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BREAKS,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COMMENT_TS,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COMPILING_ORG,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DISS_ORG,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COVERAGE,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DATA_COMP,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DECIMALS,1071509.0,2.959498,0.197133,2.0,3.000000,3.000000,3.000000,3.000000e+00
UNIT_MULT,1071509.0,5.752235,1.193818,0.0,6.000000,6.000000,6.000000,6.000000e+00


,field,nunique,top_10
0,FRD_TYP,9,"_Z (1,067,333) | F (2,352) | F15 (288) | F11 (288) | F12 (288) | F3 (288) | F10 (288) | F2 (256) | F13 (128)"
1,UNIT_MEASURE,17,"PN (505,508) | EUR (450,833) | XDF (82,358) | PN_R_POP (5,751) | EUR_R_POP (5,432) | EUR_R_PNT (5,395) | EUR_R_CP0 (4,909) | PN_R_CP0 (4,894) | EUR_R_TT (1,374) | PN_R_TT (1,369)"
2,FREQ,3,"H (954,053) | A (58,831) | Q (58,625)"
3,OBS_STATUS,6,"A (851,676) | P (114,719) | Q (85,005) | M (18,445) | E (1,178) | L (486)"
4,TYP_TRNSCTN,4,"CP0 (953,679) | CW1 (97,082) | CP1 (10,926) | CP11 (9,822)"
5,RMT_INTTN,3,"NR (376,052) | _T (363,989) | R (331,468)"
6,INTTN_CHNNL,8,"2000 (505,658) | _T (362,723) | 2223 (63,689) | 1000 (61,366) | 2229 (31,046) | 2221 (23,914) | 2230 (12,501) | 2231 (10,612)"
7,KEY,136259,
8,TITLE,5634,


A                         Обычное значение
M                     Значение отсутствует
Q                          Значение скрыто
P                          Предварительное
L    Данные существуют, но не были собраны
E                                Оценочное
Name: meaning, dtype: str

OBS_VALUE: пропусков 103,936, нулей 120,773, отрицательных 1,293.


TRANSFORMATION
G1    1293
Name: negative_rows, dtype: Int64

Проверка файла pcp.csv показала 1 071 509 строк и 35 столбцов. Периоды в исходной выгрузке охватывают интервал с 2000 года по II квартал 2026 года. По ключу KEY + TIME_PERIOD дубликатов нет. В поле OBS_VALUE обнаружено 103 936 пропусков (9,70%). 

Для исследования используются полугодовые данные за 2022–2024 годы. Целевая переменная fraud_rate рассчитывается как отношение показателя мошеннических операций (FRD_TYP = F) к показателю всех операций (FRD_TYP = _Z) в одном и том же срезе. Доли по количеству операций и по денежной сумме рассчитываются отдельно.

| Поле | Шкала | Характеристика |
|---|---|---|
| `KEY` | Номинальная (ID) | Идентификатор статистического ряда. Повторяется для разных периодов этого ряда. |
| `REF_AREA`, `COUNT_AREA`, `TRMNL_LCTN` | Категориальная | Территория отчётности, территория контрагента и местоположение терминала. |
| `TYP_TRNSCTN`, `INTTN_CHNNL`, `RMT_INTTN`, `PYMNT_SCHM`, `CRD_FNCTN`, `SCA`, `FRD_TYP` | Категориальная | Тип операции, канал, дистанционность, платёжная схема, функция карты, категория аутентификации и тип мошенничества. |
| `RL_TRNSCTN` | Дихотомическая в этой выгрузке | Две наблюдаемые роли в операции, закодированные как `1` и `2`. |
| `FREQ`, `TIME_FORMAT`, `TIME_PER_COLLECT` | Категориальная | Частота ряда, формат периода и способ отнесения значения к периоду. |
| `TRANSFORMATION`, `UNIT_MEASURE`, `UNIT` | Категориальная | Вид преобразования показателя и коды единиц измерения. |
| `OBS_STATUS`, `CONF_STATUS`, `COMPILING_ORG`, `DISS_ORG` | Категориальная | Статусы наблюдения и конфиденциальности, коды организаций-составителей и распространителей. |
| `TIME_PERIOD` | Интервальная | Год, квартал или полугодие наблюдения. В исследовании сравниваются полугодия одинаковой длительности. |
| `OBS_VALUE`, `PRE_BREAK_VALUE` | Зависит от показателя | Для исходных уровней `PN` — абсолютная шкала, поскольку считаются операции. Для `EUR` — шкала отношений, поскольку измеряется денежная сумма. `PRE_BREAK_VALUE` в этой выгрузке пуст. |
| `DECIMALS` | Абсолютная | Количество десятичных знаков, служебное поле точности представления. |
| `UNIT_MULT` | Интервальная на шкале показателя степени | Степень десяти, задающая масштаб `OBS_VALUE`. Например, `6` означает множитель \(10^6\). |
| `TITLE`, `TITLE_COMPL`, `COMMENT_OBS`, `COMMENT_TS`, `BREAKS`, `COVERAGE`, `DATA_COMP`, `METHOD_REF` | Контентная | Названия рядов, комментарии и описания разрывов, охвата и методики. Часть полей полностью пустая в этой выгрузке. |

### 2.2.2. ECB PCT — кредитовые переводы (XML)

Набор Credit transfers (including fraud data) опубликован Европейским центральным банком на ECB Data Portal. Он содержит статистику отправленных и полученных кредитовых переводов, а также мошеннических отправленных переводов. В файле pct_xml.xml одна строка после разбора XML соответствует одному наблюдению статистического ряда за один период.

- страна (REF_AREA),

- период (TIME_PERIOD, например 2023-S1),

- схема перевода (PYMNT_SCHM: CTS_ALL — все, CTS_SEPA — SEPA, CTS_SEPAI — SEPA instant, ONUS — внутрибанковские),

- канал инициирования (INTTN_CHNNL: _T — все, 2000 — электронно, 1200 — на бумаге),

- дистанционность (RMT_INTTN: _T — все, R — дистанционно, NR — не дистанционно),

- единица (UNIT_MEASURE: PN — количество переводов, EUR — сумма в евро),

- тип ряда (FRD_TYP: _Z — все операции, F — мошеннические).

In [4]:
# Разбираем XML с переводами. Если подходящего XML нет, используем CSV.
def local_name(tag):
    return tag.rsplit("}", 1)[-1]

def sdmx_attributes(element):
    return {
        child.attrib["id"]: child.get("value")
        for child in element if local_name(child.tag) == "Value"
    }

SERIES_ATTR_NODES = {"SeriesKey", "Attributes"}
OBS_FIELD_ALIASES = {"ObsDimension": "TIME_PERIOD", "Time": "TIME_PERIOD"}


def _series_attributes(series_el):
    """Собираем атрибуты серии из самого элемента, SeriesKey и Attributes."""
    attrs = dict(series_el.attrib)
    for child in series_el:
        if local_name(child.tag) in SERIES_ATTR_NODES:
            attrs.update(sdmx_attributes(child))
    return attrs


def _obs_row(obs_el, series_attrs):
    """Добавляем к атрибутам серии период, значение и атрибуты наблюдения."""
    row = {**series_attrs, **obs_el.attrib}
    for field in obs_el:
        name = local_name(field.tag)
        if name in OBS_FIELD_ALIASES:
            row[OBS_FIELD_ALIASES[name]] = field.get("value") or field.text
        elif name == "ObsValue":
            row["OBS_VALUE"] = field.get("value")
        elif name == "Attributes":
            row.update(sdmx_attributes(field))
    return row


def parse_sdmx_xml(path):
    # Разбираем XML по одной серии и собираем строки в блоки.
    rows, blocks, stack, series_fields = [], [], [], set()
    for event, element in ET.iterparse(path, events=("start", "end")):
        if event == "start":
            stack.append(element)
            continue
        if local_name(element.tag) == "Series":
            series_attrs = _series_attributes(element)
            series_fields.update(series_attrs)
            rows.extend(_obs_row(obs, series_attrs) for obs in element
                        if local_name(obs.tag) == "Obs")
            # Удаляем прочитанную серию из дерева, чтобы пустые элементы не копились.
            stack[-2].remove(element)
            element.clear()
            if len(rows) >= 50_000:
                blocks.append(pd.DataFrame.from_records(rows))
                rows.clear()
        stack.pop()

    if rows:
        blocks.append(pd.DataFrame.from_records(rows))
    if not blocks:
        raise ValueError(f"В {path.name} нет наблюдений SDMX по сериям.")

    result = pd.concat(blocks, ignore_index=True, sort=False)
    if ("PYMNT_SCHM" not in result or "TYP_TRNSCTN" in series_fields
            or not result["PYMNT_SCHM"].eq("CTS_ALL").any()):
        raise ValueError(f"{path.name}: XML не соответствует структуре PCT.")
    return type_ecb(result, PCT_DIMS), sorted(series_fields)


def select_pct_xml():
    explicit = os.environ.get("FRAUD_PCT_XML")
    candidates = [Path(explicit)] if explicit else [
        folder / name for name in ("pct_xml.xml", "data.xml") for folder in (DATA_DIR, ROOT)
    ]
    selected = None
    for path in dict.fromkeys(candidates):
        if not path.is_file():
            continue
        with path.open("rb") as stream:
            header = stream.read(8192)
        if b"ECB_PAY5" in header:
            print(f"{path.name}: структура ECB_PAY5, карточные платежи PCP.")
        elif b"ECB_PAY2" in header and selected is None:
            selected = path
        elif b"ECB_PAY2" not in header:
            print(f"{path.name}: структура PCT ECB_PAY2 не распознана.")
    return selected

pct_xml_path = select_pct_xml()
if pct_xml_path is None:
    pct_raw = read_ecb_csv(local_file("pct.csv"), PCT_DIMS)
    pct_input_format = "CSV"
    print("Использован PCT CSV. Поля серии:", ", ".join(PCT_DIMS))
else:
    pct_raw, pct_series_fields = parse_sdmx_xml(pct_xml_path)
    pct_input_format = "XML"
    print(f"Источник: {pct_xml_path.name}. Атрибуты серий XML:", ", ".join(pct_series_fields))

compact_profile(pct_raw, f"ECB PCT ({pct_input_format})", "TIME_PERIOD", PCT_DIMS + ["TIME_PERIOD"])
show_ecb_codes(pct_raw)

Источник: pct_xml.xml. Атрибуты серий XML: COUNT_AREA, DECIMALS, FRD_TYP, FREQ, INTTN_CHNNL, METHOD_REF, PYMNT_SCHM, REF_AREA, RL_TRNSCTN, RMT_INTTN, SCA, TIME_FORMAT, TIME_PER_COLLECT, TITLE, TITLE_COMPL, TRANSFORMATION, UNIT, UNIT_MEASURE, UNIT_MULT
ECB PCT (XML) · shape = (285937, 24)


,dtype,missing,missing_pct
FREQ,string,0,0.00
REF_AREA,string,0,0.00
COUNT_AREA,string,0,0.00
RL_TRNSCTN,string,0,0.00
INTTN_CHNNL,string,0,0.00
RMT_INTTN,string,0,0.00
PYMNT_SCHM,string,0,0.00
SCA,string,0,0.00
FRD_TYP,string,0,0.00
TRANSFORMATION,string,0,0.00


Диапазон: 2000 2026-Q2
Дубликаты по FREQ + REF_AREA + COUNT_AREA + RL_TRNSCTN + INTTN_CHNNL + RMT_INTTN + PYMNT_SCHM + SCA + FRD_TYP + TRANSFORMATION + UNIT_MEASURE + TIME_PERIOD = 0


,FREQ,REF_AREA,COUNT_AREA,RL_TRNSCTN,INTTN_CHNNL,RMT_INTTN,PYMNT_SCHM,SCA,FRD_TYP,TRANSFORMATION,...,TITLE_COMPL,UNIT,TITLE,UNIT_MULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,CONF_STATUS,METHOD_REF,COMMENT_OBS
0,A,AT,BE,1,1200,_T,CTS_ALL,_X,_Z,N,...,"Value of credit transfers, initiated in paper-...",EUR,"Value of credit transfers, initiated in paper-...",6,2022,733.670819,A,F,NaN,NaN
1,A,AT,BE,1,1200,_T,CTS_ALL,_X,_Z,N,...,"Value of credit transfers, initiated in paper-...",EUR,"Value of credit transfers, initiated in paper-...",6,2023,691.137581,A,F,NaN,NaN
2,A,AT,BE,1,1200,_T,CTS_ALL,_X,_Z,N,...,"Value of credit transfers, initiated in paper-...",EUR,"Value of credit transfers, initiated in paper-...",6,2024,3451.712408,A,F,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
DECIMALS,285937.0,2.962303e+00,1.904630e-01,2.0,3.000000,3.00000,3.000000,3.000000e+00
UNIT_MULT,285937.0,5.773377e+00,1.143847e+00,0.0,6.000000,6.00000,6.000000,6.000000e+00
OBS_VALUE,250332.0,1.322198e+06,2.209380e+07,-100.0,0.042425,20.41748,3672.680887,1.681416e+09


,field,nunique,top_10
0,FRD_TYP,2,"_Z (284,905) | F (1,032)"
1,UNIT_MEASURE,16,"PN (133,774) | EUR (120,561) | XDF (23,579) | EUR_R_POP (1,421) | EUR_R_TT (1,280) | EUR_R_PNT (1,279) | PN_R_TT (1,274) | PN_R_POP (1,271) | EUR_R_B1GQ (625) | XDF_R_TT (276)"
2,FREQ,3,"H (148,543) | A (77,438) | Q (59,956)"
3,OBS_STATUS,6,"A (229,661) | Q (28,667) | P (20,043) | M (6,290) | L (648) | E (628)"
4,RMT_INTTN,3,"_T (185,973) | R (51,019) | NR (48,945)"
5,INTTN_CHNNL,13,"2000 (119,904) | _T (113,905) | 1200 (10,513) | 2100 (9,641) | 2200 (9,641) | 2210 (9,631) | 2220 (2,186) | 2230 (2,186) | 2231 (2,186) | 5000 (2,058)"
6,TITLE,2122,


A                         Обычное значение
Q                          Значение скрыто
M                     Значение отсутствует
E                                Оценочное
P                          Предварительное
L    Данные существуют, но не были собраны
Name: meaning, dtype: str

OBS_VALUE: пропусков 35,605, нулей 29,525, отрицательных 530.


TRANSFORMATION
G1    530
Name: negative_rows, dtype: Int64

Проверка показала 285 937 строк и 24 столбца. Исходная выгрузка охватывает периоды с 2000 года по II квартал 2026 года. Дубликатов по комбинации измерений ряда и TIME_PERIOD нет. В OBS_VALUE обнаружено 35 605 пропусков (12,45%). Для исследования используются полугодовые данные за 2022–2024 годы. Показатель fraud_rate рассчитывается делением значения для мошеннических переводов (FRD_TYP = F) на значение для всех переводов (FRD_TYP = _Z) в том же срезе. Для PN доля рассчитывается по количеству, для EUR — по сумме. 

| Поле | Шкала | Характеристика |
|---|---|---|
| `REF_AREA`, `COUNT_AREA` | Категориальная | Территория отчётности и территория контрагента. |
| `RL_TRNSCTN` | Дихотомическая в этой выгрузке | Роль в операции, соответствующая отправленным или полученным переводам. |
| `FRD_TYP` | Дихотомическая в этой выгрузке | Два вида статистических рядов: все переводы (`_Z`) и мошеннические переводы (`F`). Общий ряд включает мошеннические переводы. |
| `INTTN_CHNNL`, `RMT_INTTN`, `PYMNT_SCHM`, `SCA` | Категориальная | Канал, дистанционность, платёжная схема и категория усиленной аутентификации. |
| `FREQ`, `TIME_FORMAT`, `TIME_PER_COLLECT` | Категориальная | Частота ряда, формат периода и способ отнесения значения к периоду. |
| `TRANSFORMATION`, `UNIT_MEASURE`, `UNIT` | Категориальная | Вид преобразования показателя и коды единиц измерения. |
| `OBS_STATUS`, `CONF_STATUS` | Категориальная | Статус наблюдения и статус конфиденциальности. |
| `TIME_PERIOD` | Интервальная | Год, квартал или полугодие наблюдения. |
| `OBS_VALUE` | Абсолютная для `PN`, отношений для `EUR` | Для исходных уровней это количество переводов или их сумма в евро. |
| `DECIMALS` | Абсолютная | Число десятичных знаков, указанное в метаданных ряда. |
| `UNIT_MULT` | Интервальная на шкале показателя степени | Степень десяти, задающая масштаб значения. Например, `6` означает множитель \(10^6\). |
| `TITLE`, `TITLE_COMPL`, `METHOD_REF`, `COMMENT_OBS` | Контентная | Названия рядов, сведения о методике и комментарии к наблюдениям. |

### 2.2.3. Eurostat — онлайн-покупки (JSON-stat)

Набор Internet purchases by individuals (isoc_ec_ib20) - это официальная статистика о том, какая доля населения покупала что-то в интернете за последние 12 месяцев. Это не мошенничество, а фактор: чем больше людей покупают онлайн, тем больше возможностей для мошенников в дистанционных операциях. Поэтому Eurostat присоединяется к fraud_rate как внешний признак, а не как целевая переменная. 

- страна (geo: DE, FR, IT, ES и т.д.),

- год (time: 2022, 2023, 2024),

- индикатор (indic_is: I_BLT12 — покупки за 12 месяцев, I_BUY3 — за 3 месяца и т.д.),

- группа населения (ind_type: IND_TOTAL — всё население, Y25_34 — возрастная группа и т.д.),

- единица (unit: PC_IND — процент от обследуемого населения),

- значение (value: число от 0 до 100).

In [5]:
def jsonstat_codes(index, size):
    if isinstance(index, dict):
        if sorted(index.values()) != list(range(size)):
            raise ValueError("Некорректные позиции категорий JSON-stat.")
        return sorted(index, key=index.get)
    if isinstance(index, list) and len(index) == size and len(set(index)) == size:
        return index
    raise ValueError("Некорректная длина измерения JSON-stat.")

def decode_jsonstat(payload):
    ids, sizes = payload["id"], payload["size"]
    if len(ids) != len(sizes) or len(set(ids)) != len(ids):
        raise ValueError("Размеры куба JSON-stat не совпадают с измерениями.")
    axes = [jsonstat_codes(payload["dimension"][d]["category"]["index"], n)
            for d, n in zip(ids, sizes)]
    index = pd.MultiIndex.from_product(axes, names=ids)
    frame = index.to_frame(index=False)
    for field in ("value", "status"):
        raw = payload.get(field, {})
        if isinstance(raw, dict):
            values = [None] * len(frame)
            for position, value in raw.items():
                offset = int(position)
                if not 0 <= offset < len(values):
                    raise ValueError(f"Позиция {offset} вне куба JSON-stat.")
                values[offset] = value
        elif isinstance(raw, list) and len(raw) == len(frame):
            values = raw
        else:
            raise ValueError(f"Некорректное поле {field} JSON-stat.")
        frame[field] = values
    frame["value"] = pd.to_numeric(frame["value"], errors="raise")
    return frame

with local_file("isoc_ec_ib20.json").open(encoding="utf-8-sig") as stream:
    euro_payload = json.load(stream)
euro_raw = decode_jsonstat(euro_payload)
euro_dims = euro_payload["id"]
display(pd.DataFrame({"dimension": euro_dims, "levels": euro_payload["size"]}))
compact_profile(euro_raw, "Eurostat", "time", euro_dims)
print("Географии:", ", ".join(sorted(euro_raw["geo"].unique())))

,dimension,levels
0,freq,1
1,ind_type,147
2,indic_is,13
3,unit,4
4,geo,40
5,time,6


Eurostat · shape = (1834560, 8)


,dtype,missing,missing_pct
freq,str,0,0.00
ind_type,str,0,0.00
indic_is,str,0,0.00
unit,str,0,0.00
geo,str,0,0.00
time,str,0,0.00
value,float64,1303722,71.06
status,str,1759556,95.91


Диапазон: 2020 2025
Дубликаты по freq + ind_type + indic_is + unit + geo + time = 0


,freq,ind_type,indic_is,unit,geo,time,value,status
0,A,IND_TOTAL,I_BUY3,PC_IND,EU27_2020,2020,53.76,e
1,A,IND_TOTAL,I_BUY3,PC_IND,EU27_2020,2021,56.94,b
2,A,IND_TOTAL,I_BUY3,PC_IND,EU27_2020,2022,56.02,NaN


,count,mean,std,min,25%,50%,75%,max
value,530838.0,25.576742,23.826633,0.0,8.56,17.18,33.77,100.0


Географии: AL, AT, BA, BE, BG, CH, CY, CZ, DE, DK, EA, EE, EL, ES, EU27_2020, FI, FR, HR, HU, IE, IS, IT, LT, LU, LV, ME, MK, MT, NL, NO, PL, PT, RO, RS, SE, SI, SK, TR, UK, XK



Проверка показала 1 834 560 строк и 8 столбцов за 2020–2025 годы. Из них 530 838 строк содержат числовое значение. В остальных 71,06% строк значение отсутствует, поскольку при преобразовании JSON в таблицу были сохранены все комбинации измерений, включая те, для которых Eurostat не опубликовал число. Дубликатов по комбинации шести измерений нет.

Для исследования выбран показатель online_purchase_share — доля обследуемого населения, совершавшего онлайн-покупки за последние 12 месяцев. Для этого используются коды I_BLT12 (покупки за 12 месяцев), IND_TOTAL (все обследуемые лица) и PC_IND (процент от обследуемого населения). За 2022–2024 годы в отобранной таблице заполнены все 78 сочетаний «страна × год». Годовое значение присоединяется к обоим полугодиям соответствующей страны и используется как объясняющий признак для fraud_rate.

| Поле | Шкала | Характеристика |
|---|---|---|
| `freq` | Категориальная | Частота наблюдений. В выгрузке встречается только `A` — годовая. |
| `ind_type` | Категориальная | Группа населения. Для исследования выбрана общая категория `IND_TOTAL`. |
| `indic_is` | Категориальная | Вид показателя онлайн-покупок, например покупки за последние 3 или 12 месяцев. |
| `unit` | Категориальная | Единица измерения. Для выбранного показателя используется `PC_IND`. |
| `geo` | Категориальная | Код страны или группы стран. |
| `time` | Интервальная | Календарный год наблюдения. |
| `value` | Отношений для выбранного показателя | Доля совершавших онлайн-покупки, выраженная в процентах. После отбора поле переименовано в `online_purchase_share`. |
| `status` | Категориальная | Дополнительная отметка Eurostat о значении, например об оценочном характере. После отбора поле переименовано в `online_eurostat_flag`. |


### 2.2.4. EBA/ECB — регуляторные публикации (HTML)

Корпус включает [решение EBA об отчётности](https://www.eba.europa.eu/publications-and-media/press-releases/eba-adopts-decision-reporting-payment-fraud-data-under), [релиз ECB о карточном мошенничестве](https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.pr230526~f09bc3c664.en.html), [заключение EBA о новых схемах](https://www.eba.europa.eu/publications-and-media/press-releases/eba-has-identified-new-types-payment-fraud-and-proposes-measures-mitigate-underlying-risks-and) и [совместный релиз ECB/EBA](https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.pr240801~f21cc4a009.en.html).


- published — дата публикации (2022-07-01 … 2024-08-01),

- agency — кто опубликовал (EBA, ECB, ECB/EBA),

- topic — тема (reporting, card_fraud, fraud_types, joint_report),

- url, html_file — ссылка и имя сохранённого файла (ID),

- characters, words — длина текста (символы, слова),

- sca_mentions, fraud_mentions, remote_mentions, remote_payment_mentions, cross_border_mentions — сколько раз встретились ключевые термины,

- text_preview — короткий фрагмент текста для просмотра,

- text_mode — snapshot (зафиксированные счётчики) или html (пересчёт по сохранённой странице).

In [6]:
REFRESH_HTML = False
ARTICLES = [
    ("2022-07-01", "EBA", "reporting",
     "https://www.eba.europa.eu/publications-and-media/press-releases/eba-adopts-decision-reporting-payment-fraud-data-under"),
    ("2023-05-26", "ECB", "card_fraud",
     "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.pr230526~f09bc3c664.en.html"),
    ("2024-04-29", "EBA", "fraud_types",
     "https://www.eba.europa.eu/publications-and-media/press-releases/eba-has-identified-new-types-payment-fraud-and-proposes-measures-mitigate-underlying-risks-and"),
    ("2024-08-01", "ECB/EBA", "joint_report",
     "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.pr240801~f21cc4a009.en.html"),
]
TERM_PATTERNS = {
    "sca_mentions": r"\b(?:strong customer authentication|SCA)\b",
    "fraud_mentions": r"\bfraud(?:s|ulent|ster(?:s)?)?\b",
    "remote_mentions": r"\bremote(?:ly)?\b",
    "remote_payment_mentions": r"\bremote[\s-]+(?:card[\s-]+)?payments?\b",
    "cross_border_mentions": r"\bcross[\s‐‑–-]+border\b",
}
TEXT_METRICS = ["characters", "words", *TERM_PATTERNS]
# Длина и частоты измерены по основным абзацам официальных публикаций.
SNAPSHOT = dict(zip([row[2] for row in ARTICLES], [[661,101,0,3,0,0,0],[2230,359,1,19,1,0,2],[1378,210,2,7,0,0,0],[2040,327,5,12,0,0,1]]))
SUMMARIES = {
    "reporting": "EBA описывает порядок передачи сведений о мошенничестве по PSD2 и обмен данными между национальными органами, EBA и ECB.",
    "card_fraud": "ECB сравнивает карточное мошенничество в 2020 и 2021 годах, включая дистанционные операции и трансграничные платежи.",
    "fraud_types": "EBA рассматривает адаптацию мошенников к SCA, социальную инженерию и дополнительные меры защиты.",
    "joint_report": "ECB и EBA сравнивают мошенничество по платёжным инструментам, а также рассматривают SCA и географию контрагента.",
}

def text_metrics(body):
    return {
        "characters": len(body),
        "words": len(re.findall(r"\w+(?:[-’']\w+)*", body)),
        **{field: len(re.findall(pattern, body, flags=re.I))
           for field, pattern in TERM_PATTERNS.items()},
    }

def load_article(url, filename):
    path = DATA_DIR / "html" / filename
    if path.is_file():
        return path.read_bytes()
    response = requests.get(
        url, headers={"User-Agent": "Mozilla/5.0 (educational research)"},
        timeout=(10, 35),
    )
    response.raise_for_status()
    return response.content

def clean_article(html):
    soup = BeautifulSoup(html, "html.parser")
    scope = soup.find("main") or soup.find("article") or soup
    for node in scope.select("script, style, nav, footer, header, aside, form, "
                             "figure, figcaption, sup, [role='navigation']"):
        node.decompose()
    body = []
    for node in scope.find_all(["p", "h2", "h3", "h4", "h5", "h6"]):
        line = " ".join(node.get_text(" ", strip=True).split())
        line = re.sub(r"\s+([,.;:!?%)])", r"\1", line)
        if not body and not line.startswith(
            ("The European Banking Authority", "The European Central Bank")
        ):
            continue
        if re.match(r"^(?:Legal basis|Background|Documents|Press contacts|Contact\b|For media queries)",
                    line, flags=re.I):
            break
        if node.name != "p" or re.match(
            r"^(?:Source:|Note:|Total value and volume|Absolute and relative levels|\(left-hand scale)",
            line, flags=re.I,
        ):
            continue
        if line:
            body.append(line)
    cleaned = " ".join(body)
    if len(cleaned) < 250:
        raise ValueError("Основной текст HTML не распознан. Проверьте сохранённую страницу.")
    return cleaned

records = []
for published, agency, topic, url in ARTICLES:
    filename = f"{published}_{topic}.html"
    record = {"published": pd.Timestamp(published), "agency": agency,
              "topic": topic, "url": url, "html_file": filename}
    if REFRESH_HTML:
        body = clean_article(load_article(url, filename))
        record.update(text_metrics(body), text_preview=body[:200], text_mode="html")
    else:
        record.update(dict(zip(TEXT_METRICS, SNAPSHOT[topic])),
                      text_preview=SUMMARIES[topic], text_mode="snapshot")
    records.append(record)

corpus = pd.DataFrame.from_records(records)
assert len(corpus) == 4 and corpus["url"].is_unique
compact_profile(corpus, "EBA/ECB", "published", ["url"])
with pd.option_context("display.max_colwidth", 200):
    display(corpus[["published", "agency", "topic", *TEXT_METRICS, "text_preview"]])

EBA/ECB · shape = (4, 14)


,dtype,missing,missing_pct
published,datetime64[us],0,0.0
agency,str,0,0.0
topic,str,0,0.0
url,str,0,0.0
html_file,str,0,0.0
characters,int64,0,0.0
words,int64,0,0.0
sca_mentions,int64,0,0.0
fraud_mentions,int64,0,0.0
remote_mentions,int64,0,0.0


Диапазон: 2022-07-01 00:00:00 2024-08-01 00:00:00
Дубликаты по url = 0


,published,agency,topic,url,html_file,characters,words,sca_mentions,fraud_mentions,remote_mentions,remote_payment_mentions,cross_border_mentions,text_preview,text_mode
0,2022-07-01,EBA,reporting,https://www.eba.europa.eu/publications-and-med...,2022-07-01_reporting.html,661,101,0,3,0,0,0,EBA описывает порядок передачи сведений о моше...,snapshot
1,2023-05-26,ECB,card_fraud,https://www.ecb.europa.eu/press/pr/date/2023/h...,2023-05-26_card_fraud.html,2230,359,1,19,1,0,2,ECB сравнивает карточное мошенничество в 2020 ...,snapshot
2,2024-04-29,EBA,fraud_types,https://www.eba.europa.eu/publications-and-med...,2024-04-29_fraud_types.html,1378,210,2,7,0,0,0,"EBA рассматривает адаптацию мошенников к SCA, ...",snapshot


,count,mean,std,min,25%,50%,75%,max
characters,4.0,1577.25,711.672853,661.0,1198.75,1709.0,2087.50,2230.0
words,4.0,249.25,117.769195,101.0,182.75,268.5,335.00,359.0
sca_mentions,4.0,2.00,2.160247,0.0,0.75,1.5,2.75,5.0
fraud_mentions,4.0,10.25,6.898067,3.0,6.00,9.5,13.75,19.0
remote_mentions,4.0,0.25,0.500000,0.0,0.00,0.0,0.25,1.0
remote_payment_mentions,4.0,0.00,0.000000,0.0,0.00,0.0,0.00,0.0
cross_border_mentions,4.0,0.75,0.957427,0.0,0.00,0.5,1.25,2.0


,published,agency,topic,characters,words,sca_mentions,fraud_mentions,remote_mentions,remote_payment_mentions,cross_border_mentions,text_preview
0,2022-07-01,EBA,reporting,661,101,0,3,0,0,0,"EBA описывает порядок передачи сведений о мошенничестве по PSD2 и обмен данными между национальными органами, EBA и ECB."
1,2023-05-26,ECB,card_fraud,2230,359,1,19,1,0,2,"ECB сравнивает карточное мошенничество в 2020 и 2021 годах, включая дистанционные операции и трансграничные платежи."
2,2024-04-29,EBA,fraud_types,1378,210,2,7,0,0,0,"EBA рассматривает адаптацию мошенников к SCA, социальную инженерию и дополнительные меры защиты."
3,2024-08-01,ECB/EBA,joint_report,2040,327,5,12,0,0,1,"ECB и EBA сравнивают мошенничество по платёжным инструментам, а также рассматривают SCA и географию контрагента."


Проверка показала 4 строки и 14 столбцов. Публикации датированы периодом с 1 июля 2022 года по 1 августа 2024 года. Пропусков в таблице и повторов по url нет. Для основных абзацев подсчитано число слов, символов и упоминаний пяти групп терминов. В сохранённом запуске использованы ранее рассчитанные счётчики, а text_preview содержит краткий пересказ каждой публикации. Аудит в ноутбуке

Для объединения с платёжными данными публикации группируются по году выхода. По каждому году рассчитаны суммарные счётчики и частоты упоминаний на 1 000 слов. Эти признаки описывают содержание выбранных публикаций. Всего в корпусе три годовых наблюдения, поэтому текстовые признаки используются как контекст исследования.

| Поле | Шкала | Характеристика |
|---|---|---|
| `url`, `html_file` | Номинальная (ID) | Адрес публикации и имя соответствующего HTML-файла. |
| `published` | Интервальная | Дата публикации. Из неё извлекается год для объединения с платёжными данными. |
| `agency`, `topic` | Категориальная | Организация-издатель и тема публикации. |
| `text_mode` | Дихотомическая по настройке | Источник счётчиков: сохранённые результаты (`snapshot`) или повторный расчёт по HTML (`html`). |
| `text_preview` | Контентная | Краткий пересказ в режиме `snapshot` или фрагмент очищенного оригинала в режиме `html`. |
| `characters`, `words` | Абсолютная | Число символов и слов в анализируемом тексте. |
| `sca_mentions`, `fraud_mentions`, `remote_mentions`, `remote_payment_mentions`, `cross_border_mentions` | Абсолютная | Число совпадений с шаблонами для усиленной аутентификации, мошенничества, дистанционности, дистанционных платежей и трансграничных операций. |

## 2.3. Подготовка данных к анализу

### 2.3.1. Подготовка ECB PC

Из набора ECB PCP отобраны полугодовые данные за 2022–2024 годы по выбранным странам. В выборку вошли четыре типа операций (CP0, CW1, CP1, CP11). Количество операций (PN) и их сумма в евро (EUR) анализируются отдельно.

Для расчёта доли мошенничества сопоставлены два ряда: все операции (_Z) и мошеннические операции (F). Для каждой пары совпадают страна, полугодие, тип операции и остальные характеристики, включая канал и дистанционность. Перед делением значения приводятся к исходным единицам с учётом масштаба UNIT_MULT. Подтипы мошенничества, например F10, отдельно не суммируются, поскольку для расчёта используется общий ряд F, а также чтобы избежать двойного счёта.

Если одно из значений отсутствует, общее количество или сумма равны нулю либо показатель мошенничества выходит за допустимые пределы, доля остаётся пустой. Необычно высокие, но допустимые доли проверяются отдельно. Общие итоги и детальные категории рассматриваются раздельно, чтобы одни и те же операции не учитывались дважды.


In [7]:
def study_levels(frame):
    scope = (
        frame["FREQ"].eq("H")
        & frame["REF_AREA"].isin(COUNTRIES)
        & frame["TRANSFORMATION"].eq("N")
        & frame["UNIT_MEASURE"].isin(["PN", "EUR"])
        & frame["FRD_TYP"].isin(["_Z", "F"])
    )
    if "TYP_TRNSCTN" in frame:
        scope &= frame["TYP_TRNSCTN"].isin(["CP0", "CW1", "CP1", "CP11"])
    out = frame.loc[scope].copy()
    out["period"] = out["TIME_PERIOD"].str.replace(
        r"^(\d{4})-[SH]([12])$", r"\1-H\2", regex=True
    )
    return out.loc[out["period"].isin(HALVES)].copy()

def paired_fraud(frame, dims, source):
    rows = study_levels(frame)
    keys = ["period", *[d for d in dims if d != "FRD_TYP"]]
    pair_key = keys + ["FRD_TYP"]
    if rows[pair_key].isna().any().any() or rows.duplicated(pair_key).any():
        raise ValueError(f"{source}: неполный ключ или дубликаты пар.")
    if "OBS_STATUS" not in rows:
        rows["OBS_STATUS"] = pd.NA
    unreported = rows["OBS_STATUS"].isin(["Q", "M", "L", "H", "O"])
    values = rows["OBS_VALUE"].astype(float) * np.power(10.0, rows["UNIT_MULT"].astype(float))
    rows["amount"] = values.where(np.isfinite(values) & ~unreported)
    cols = ["amount", "OBS_STATUS", "UNIT_MULT"]
    total = rows.loc[rows["FRD_TYP"].eq("_Z"), keys + cols].rename(columns={
        "amount": "total", "OBS_STATUS": "total_obs_status", "UNIT_MULT": "total_unit_mult",
    })
    fraud = rows.loc[rows["FRD_TYP"].eq("F"), keys + cols].rename(columns={
        "amount": "fraud", "OBS_STATUS": "fraud_obs_status", "UNIT_MULT": "fraud_unit_mult",
    })
    paired = total.merge(fraud, on=keys, how="outer", validate="one_to_one", indicator=True)
    status = np.select(
        [
            paired["_merge"].eq("left_only"), paired["_merge"].eq("right_only"),
            paired["total"].isna(), paired["fraud"].isna(),
            paired["total"].le(0), paired["fraud"].lt(0), paired["fraud"].gt(paired["total"]),
        ],
        [
            "no_fraud_series", "no_total_series", "missing_total", "missing_fraud",
            "nonpositive_total", "negative_fraud", "fraud_exceeds_total",
        ],
        default="ok",
    )
    paired["total_series_available"] = paired["_merge"].ne("right_only")
    paired["fraud_series_available"] = paired["_merge"].ne("left_only")
    paired["pair_status"] = status
    paired["fraud_rate"] = paired["fraud"].div(paired["total"].where(status == "ok"))
    paired["source"] = source
    paired["operation_type"] = {"PCP": "card_operation", "PCT": "credit_transfer"}[source]
    if source == "PCT":
        paired["TYP_TRNSCTN"] = "CT"
    return paired.drop(columns="_merge")

def pair_audit(pairs, label):
    print(f"{label} · {len(pairs):,} срезов")
    display(pd.crosstab(pairs["UNIT_MEASURE"], pairs["pair_status"]))
    display(pairs[["total", "fraud", "fraud_rate"]].isna().sum().rename("missing"))
    display(pairs[["total", "fraud", "fraud_rate"]].eq(0).sum().rename("zeros"))
    rates = pairs.loc[pairs["fraud_rate"].notna()]
    quartiles = rates.groupby("UNIT_MEASURE")["fraud_rate"].quantile([0.25, 0.75]).unstack()
    if not quartiles.empty:
        upper = quartiles[0.75] + 1.5 * (quartiles[0.75] - quartiles[0.25])
        marked = rates["fraud_rate"].gt(rates["UNIT_MEASURE"].map(upper).astype(float))
        display(pd.concat([upper.rename("iqr_upper"),
                           marked.groupby(rates["UNIT_MEASURE"]).sum().rename("flagged")], axis=1))
        display(rates.groupby("UNIT_MEASURE")["fraud_rate"].describe())

pcp_pairs = paired_fraud(pcp_raw, PCP_DIMS, "PCP")
pair_audit(pcp_pairs, "PCP, все выбранные уровни")
display(pcp_pairs.loc[pcp_pairs["fraud_rate"].notna(),
                      ["REF_AREA", "period", "TYP_TRNSCTN", "INTTN_CHNNL",
                       "RMT_INTTN", "UNIT_MEASURE", "total", "fraud", "fraud_rate"]].head(6))

PCP, все выбранные уровни · 600,788 срезов


pair_status,missing_fraud,missing_total,no_fraud_series,ok
UNIT_MEASURE,,,,
EUR,12,5,280896,331
PN,13,7,319128,396


total          49115
fraud         600059
fraud_rate    600061
Name: missing, dtype: int64

total         81377
fraud             0
fraud_rate        0
Name: zeros, dtype: int64

,iqr_upper,flagged
UNIT_MEASURE,,
EUR,0.000654,18
PN,0.000261,35


,count,mean,std,min,25%,50%,75%,max
UNIT_MEASURE,,,,,,,,
EUR,331.0,0.000257,0.000467,1.607404e-06,0.000047,0.000159,0.000289,0.005056
PN,396.0,0.002369,0.020897,3.296813e-07,0.000011,0.000053,0.000111,0.222452


,REF_AREA,period,TYP_TRNSCTN,INTTN_CHNNL,RMT_INTTN,UNIT_MEASURE,total,fraud,fraud_rate
2718,AT,2022-H1,CP0,_T,_T,EUR,3.031149e+10,7023471.46,0.000232
2719,AT,2022-H1,CP0,_T,_T,PN,6.899109e+08,73755.00,0.000107
2780,AT,2022-H1,CP0,_T,_T,EUR,2.547984e+10,996272.93,0.000039
2781,AT,2022-H1,CP0,_T,_T,PN,6.205536e+08,10721.00,0.000017
2792,AT,2022-H1,CW1,_T,_T,EUR,2.146843e+10,160542.30,0.000007
2793,AT,2022-H1,CW1,_T,_T,PN,9.642061e+07,470.00,0.000005


### 2.3.2. Подготовка ECB PCT

К переводам применяется та же формула, а набор измерений берётся из структуры PCT. Для общего формата добавляется аналитический код `CT`. При наличии корректного XML он используется для расчёта. В резервном режиме используется PCT CSV. При наличии `pct.csv` код сверяет значения и охват выбранных наблюдений двух файлов по полному ключу.

In [8]:
pct_pairs = paired_fraud(pct_raw, PCT_DIMS, "PCT")
pair_audit(pct_pairs, "PCT, все выбранные уровни")

def crosscheck_pct(csv_path, xml_frame):
    keys = ["period", *PCT_DIMS]
    fields = keys + ["OBS_VALUE", "UNIT_MULT"]
    csv_level = study_levels(read_ecb_csv(csv_path, PCT_DIMS))[fields]
    xml_level = study_levels(xml_frame)[fields]
    comparison = xml_level.merge(
        csv_level, on=keys, how="outer", suffixes=("_xml", "_csv"),
        indicator=True, validate="one_to_one",
    )
    both = comparison.loc[comparison["_merge"].eq("both")]
    value_match = np.isclose(
        both["OBS_VALUE_xml"], both["OBS_VALUE_csv"], rtol=1e-10, atol=1e-12, equal_nan=True
    )
    mult_match = np.isclose(both["UNIT_MULT_xml"], both["UNIT_MULT_csv"], equal_nan=True)
    print("Сверка PCT. XML без CSV, CSV без XML и пары:")
    display(comparison["_merge"].value_counts())
    print(f"Пар с расхождением значения или масштаба: {int((~(value_match & mult_match)).sum())}")

if pct_input_format == "XML":
    try:
        pct_csv_path = local_file("pct.csv")
    except FileNotFoundError:
        print("Сверка пропущена: pct.csv отсутствует.")
    else:
        crosscheck_pct(pct_csv_path, pct_raw)
else:
    print("Расчёт по PCT CSV. Для сверки положите рядом отдельный PCT XML.")

PCT, все выбранные уровни · 92,760 срезов


pair_status,missing_fraud,missing_total,no_fraud_series,ok
UNIT_MEASURE,,,,
EUR,6,6,43405,104
PN,6,2,49099,132


total         12161
fraud         92518
fraud_rate    92524
Name: missing, dtype: int64

total         18826
fraud             0
fraud_rate        0
Name: zeros, dtype: int64

,iqr_upper,flagged
UNIT_MEASURE,,
EUR,0.000043,8
PN,0.000051,4


,count,mean,std,min,25%,50%,75%,max
UNIT_MEASURE,,,,,,,,
EUR,104.0,0.000019,0.000018,0.000003,0.000008,0.000012,0.000022,0.000104
PN,132.0,0.000021,0.000013,0.000003,0.000012,0.000018,0.000028,0.000093


Сверка PCT. XML без CSV, CSV без XML и пары:


_merge
both          93016
left_only         0
right_only        0
Name: count, dtype: int64

Пар с расхождением значения или масштаба: 0


### 2.3.3. Подготовка Eurostat

Для каждой пары «страна × год» выбирается одна опубликованная оценка для всего населения. Значения по возрастам и другим группам имеют собственные знаменатели, поэтому усреднять их для построения общего показателя нельзя. `online_purchase_share` выражается в процентах от обследуемого населения и сохраняет флаг Eurostat.

При объединении код Eurostat `EL` приводится к общему коду Греции `GR`(Eurostat использует свой код для Греции — EL).

In [9]:
euro_selection = {
    "freq": "A", "ind_type": "IND_TOTAL", "indic_is": "I_BLT12", "unit": "PC_IND",
}
euro_country = euro_raw["geo"].replace({"EL": "GR"})
euro_mask = euro_country.isin(COUNTRIES) & euro_raw["time"].isin([str(y) for y in YEARS])
for col, code in euro_selection.items():
    if col not in euro_raw or code not in set(euro_raw[col]):
        raise ValueError(f"Eurostat: отсутствует {col}={code}.")
    euro_mask &= euro_raw[col].eq(code)

online = euro_raw.loc[euro_mask, ["geo", "time", "value", "status"]].rename(columns={
    "geo": "country", "time": "year", "value": "online_purchase_share", "status": "online_eurostat_flag",
})
online["country"] = online["country"].replace({"EL": "GR"})
online["year"] = online["year"].astype(int)
assert not online.duplicated(["country", "year"]).any()
assert online["online_purchase_share"].dropna().between(0, 100).all()
print(f"Годовых значений: {online['online_purchase_share'].notna().sum()} из {len(COUNTRIES) * len(YEARS)}.")
display(online.sort_values(["country", "year"]))

Годовых значений: 78 из 78.


,country,year,online_purchase_share,online_eurostat_flag
3008,AT,2022,65.65,NaN
3009,AT,2023,72.17,NaN
3010,AT,2024,70.73,NaN
2894,BE,2022,74.83,NaN
2895,BE,2023,75.26,NaN
...,...,...,...,...
3033,SI,2023,65.85,NaN
3034,SI,2024,66.26,NaN
3038,SK,2022,76.72,NaN
3039,SK,2023,76.82,NaN


### 2.3.4. Подготовка текстовых признаков

Используется словарный подсчёт по терминам SCA, fraud, remote, remote payment и cross-border. Он выбран из двух предусмотренных вариантов, поскольку в корпусе всего четыре документа. Календарный год определяется датой публикации. Годовая таблица содержит число публикаций, суммы частот и нормированные частоты на 1 000 слов. Год публикации может отличаться от года статистики, описанной в тексте.

In [10]:
text_year = (
    corpus.assign(year=corpus["published"].dt.year)
    .loc[lambda df: df["year"].isin(YEARS)]
    .groupby("year", as_index=False)
    .agg(text_publications=("url", "size"), **{
        f"text_{col}": (col, "sum") for col in TEXT_METRICS
    })
)
for term in TERM_PATTERNS:
    text_year[f"text_{term}_per_1000"] = (
        1000 * text_year[f"text_{term}"].div(text_year["text_words"].where(text_year["text_words"].gt(0)))
    )
assert text_year["year"].is_unique
display(text_year)

,year,text_publications,text_characters,text_words,text_sca_mentions,text_fraud_mentions,text_remote_mentions,text_remote_payment_mentions,text_cross_border_mentions,text_sca_mentions_per_1000,text_fraud_mentions_per_1000,text_remote_mentions_per_1000,text_remote_payment_mentions_per_1000,text_cross_border_mentions_per_1000
0,2022,1,661,101,0,3,0,0,0,0.000000,29.702970,0.000000,0.0,0.000000
1,2023,1,2230,359,1,19,1,0,2,2.785515,52.924791,2.785515,0.0,5.571031
2,2024,2,3418,537,7,19,0,0,1,13.035382,35.381750,0.000000,0.0,1.862197


### 2.3.5. Детальная полугодовая панель

Минимальный срез — страна × полугодие × тип операции × дистанционность × единица. Дополнительно сохраняются канал, функция карты, SCA и другие измерения, которые отличают ряды ECB, включая географию контрагента, географию терминала и роль. Значения, применимые только к PCP, у PCT остаются пустыми. Ряды PCP и PCT объединяются вертикально, единицы `PN` и `EUR` остаются отдельными строками. Искусственная сетка комбинаций не строится.

Годовой Eurostat присоединяется по стране и году. Годовые текстовые признаки присоединяются по году. Факторы присоединяются к срезам с рядом F. Полугодовой ключ сохраняется, число строк проверяется после каждого объединения. Оба годовых показателя повторяются в пределах года, поэтому их вариация определяется исходной годовой детализацией. Для прогноза нужно учитывать дату, когда показатель стал доступен.

`detailed_panel` содержит все срезы из выгрузок. В экспорт входят срезы, для которых существует ряд `F`, включая скрытые значения для аудита. `model_rows` содержит строки с вычисленной долей. Число строк с целевой переменной и покрытие каждого признака проверяются по данным.


In [11]:
DETAIL_KEY = [
    "source", "REF_AREA", "TIME_PERIOD", "TYP_TRNSCTN", "RMT_INTTN", "UNIT_MEASURE",
    "INTTN_CHNNL", "CRD_FNCTN", "SCA", "COUNT_AREA", "TRMNL_LCTN", "RL_TRNSCTN",
    "PYMNT_SCHM", "FREQ", "TRANSFORMATION",
]

def checked_left(left, right, keys, label):
    before = left[DETAIL_KEY].astype("string").fillna("<missing>").to_numpy(dtype=object)
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    after = merged[DETAIL_KEY].astype("string").fillna("<missing>").to_numpy(dtype=object)
    if not np.array_equal(after, before):
        raise ValueError(f"{label}: изменились ключи или число строк.")
    print(f"{label}: строк до и после {len(left):,}, без соответствия {merged['_merge'].eq('left_only').sum():,}.")
    return merged.drop(columns="_merge")

detailed_panel = pd.concat([pcp_pairs, pct_pairs], ignore_index=True).rename(
    columns={"period": "TIME_PERIOD"},
)
assert not detailed_panel.duplicated(DETAIL_KEY).any()
detailed_panel["year"] = detailed_panel["TIME_PERIOD"].str[:4].astype(int)
detailed_panel["half"] = detailed_panel["TIME_PERIOD"].str[-1].astype(int)
detailed_panel["fraud_rate_available"] = detailed_panel["fraud_rate"].notna()
detailed_panel["in_original_scope"] = detailed_panel["REF_AREA"].isin(CORE_COUNTRIES)

valid = detailed_panel["fraud_rate_available"]
assert valid.eq(detailed_panel["pair_status"].eq("ok")).all()
assert detailed_panel.loc[valid, "fraud_rate"].between(0, 1).all()
assert detailed_panel.loc[valid, "total"].gt(0).all()
for field in ("total_obs_status", "fraud_obs_status"):
    assert not detailed_panel.loc[valid, field].isin(["M", "L", "Q", "H", "O"]).any()

print(f"Всего детальных срезов: {len(detailed_panel):,}. Из них с рядом F: "
      f"{detailed_panel['fraud_series_available'].sum():,}, с вычисленной долей: {valid.sum():,}.")
display(detailed_panel.groupby(["source", "TYP_TRNSCTN", "UNIT_MEASURE"], dropna=False)
        .agg(slices=("fraud_rate", "size"), fraud_series=("fraud_series_available", "sum"),
             valid_target=("fraud_rate_available", "sum")).reset_index())

for feature in ("RMT_INTTN", "INTTN_CHNNL", "CRD_FNCTN", "SCA"):
    print(f"Покрытие {feature}")
    display(detailed_panel.groupby(["source", feature], dropna=False)
            .agg(slices=("fraud_rate", "size"), valid_target=("fraud_rate_available", "sum")).reset_index())

display(detailed_panel.groupby("in_original_scope").agg(
    slices=("fraud_rate", "size"), fraud_series=("fraud_series_available", "sum"),
    valid_target=("fraud_rate_available", "sum"),
))

# Пары R/NR с одинаковыми остальными измерениями дают сопоставимые наблюдения.
remote = detailed_panel.loc[valid & detailed_panel["RMT_INTTN"].isin(["R", "NR"])]
remote_key = [key for key in DETAIL_KEY if key != "RMT_INTTN"]
remote_matches = remote.groupby(remote_key, dropna=False)["RMT_INTTN"].nunique().eq(2)
print(f"Сопоставимых пар R/NR с вычисленной долей в обоих срезах: {remote_matches.sum():,}.")


Всего детальных срезов: 693,548. Из них с рядом F: 1,020, с вычисленной долей: 963.


,source,TYP_TRNSCTN,UNIT_MEASURE,slices,fraud_series,valid_target
0,PCP,CP0,EUR,246411,232,220
1,PCP,CP0,PN,278274,277,263
2,PCP,CP1,EUR,3490,0,0
3,PCP,CP1,PN,3495,0,0
4,PCP,CP11,EUR,2900,0,0
5,PCP,CP11,PN,3495,0,0
6,PCP,CW1,EUR,28443,116,111
7,PCP,CW1,PN,34280,139,133
8,PCT,CT,EUR,43521,116,104
9,PCT,CT,PN,49239,140,132


Покрытие RMT_INTTN


,source,RMT_INTTN,slices,valid_target
0,PCP,NR,218115,0
1,PCP,R,204702,0
2,PCP,_T,177971,727
3,PCT,NR,27781,0
4,PCT,R,29073,0
5,PCT,_T,35906,236


Покрытие INTTN_CHNNL


,source,INTTN_CHNNL,slices,valid_target
0,PCP,1000,40140,0
1,PCP,2000,308241,0
2,PCP,2221,13380,0
3,PCP,2223,26768,0
4,PCP,2229,20365,0
5,PCP,2230,6985,0
6,PCP,2231,6985,0
7,PCP,_T,177924,727
8,PCT,1200,1408,0
9,PCT,2000,61172,0


... (2 outputs left out)

Покрытие SCA


,source,SCA,slices,valid_target
0,PCP,100,26760,0
1,PCP,200,26760,0
2,PCP,201,12790,0
3,PCP,202,12790,0
4,PCP,204,12790,0
5,PCP,205,25580,0
6,PCP,206,12790,0
7,PCP,207,6395,0
8,PCP,208,12790,0
9,PCP,209,12790,0


,slices,fraud_series,valid_target
in_original_scope,,,
False,599340,876,819
True,94208,144,144


Сопоставимых пар R/NR с вычисленной долей в обоих срезах: 0.


In [12]:
# Подвыборка для экспорта и 14 кандидатов в признаки.
# Числитель, знаменатель и флаги качества исключены из списка признаков.
MODEL_FEATURES = [
    "REF_AREA", "year", "half", "TYP_TRNSCTN", "RMT_INTTN", "UNIT_MEASURE", "INTTN_CHNNL",
    "CRD_FNCTN", "SCA", "COUNT_AREA", "RL_TRNSCTN", "online_purchase_share",
    "text_sca_mentions_per_1000", "text_fraud_mentions_per_1000",
]
panel = detailed_panel.loc[detailed_panel["fraud_series_available"]].copy()
panel = checked_left(
    panel, online.rename(columns={"country": "REF_AREA"}), ["REF_AREA", "year"], "Eurostat",
)
panel = checked_left(panel, text_year, ["year"], "EBA/ECB")
panel["text_context_mode"] = corpus["text_mode"].iloc[0]
panel["ecb_input_format"] = np.where(panel["source"].eq("PCP"), "CSV", pct_input_format)
panel = panel.sort_values(DETAIL_KEY).reset_index(drop=True)
model_rows = panel.loc[panel["fraud_rate_available"]].copy()
model_X = model_rows[MODEL_FEATURES]
model_y = model_rows["fraud_rate"]
assert len(model_X) == len(model_y) and model_y.notna().all()

summary = model_X.agg(["count", "nunique"]).T
summary["missing"] = len(model_X) - summary["count"]
print("Изменчивость кандидатов в признаки среди строк с целевой переменной:")
display(summary)
display(panel[["REF_AREA", "TIME_PERIOD", "source", "TYP_TRNSCTN", "UNIT_MEASURE",
               "RMT_INTTN", "INTTN_CHNNL", "total", "fraud", "fraud_rate", "pair_status"]].head(8))

front = DETAIL_KEY + ["fraud_rate", "pair_status", "total", "fraud"]
panel = panel[front + [c for c in panel if c not in front]]
csv_path = ROOT / "fraud_panel_2022_2024.csv"
panel.to_csv(csv_path, index=False, encoding="utf-8-sig")
round_trip = pd.read_csv(csv_path, dtype={c: "string" for c in DETAIL_KEY})
assert round_trip.shape == panel.shape
assert np.array_equal(
    round_trip[DETAIL_KEY].astype("string").fillna("<missing>").to_numpy(dtype=object),
    panel[DETAIL_KEY].astype("string").fillna("<missing>").to_numpy(dtype=object),
)
assert np.allclose(round_trip["fraud_rate"], panel["fraud_rate"], equal_nan=True)
print(f"Сохранено {len(panel):,} строк × {len(panel.columns)} полей; с целевой переменной "
      f"{len(model_rows):,}.")
print(f"Полный путь к CSV: {csv_path.resolve()}")


Eurostat: строк до и после 1,020, без соответствия 0.
EBA/ECB: строк до и после 1,020, без соответствия 0.
Изменчивость кандидатов в признаки среди строк с целевой переменной:


,count,nunique,missing
REF_AREA,963,24,0
year,963,3,0
half,963,2,0
TYP_TRNSCTN,963,3,0
RMT_INTTN,963,1,0
UNIT_MEASURE,963,2,0
INTTN_CHNNL,963,1,0
CRD_FNCTN,727,1,236
SCA,963,1,0
COUNT_AREA,963,1,0


,REF_AREA,TIME_PERIOD,source,TYP_TRNSCTN,UNIT_MEASURE,RMT_INTTN,INTTN_CHNNL,total,fraud,fraud_rate,pair_status
0,AT,2022-H1,PCP,CP0,EUR,_T,_T,3.031149e+10,7023471.46,0.000232,ok
1,AT,2022-H1,PCP,CP0,EUR,_T,_T,2.547984e+10,996272.93,0.000039,ok
2,AT,2022-H1,PCP,CP0,PN,_T,_T,6.899109e+08,73755.00,0.000107,ok
3,AT,2022-H1,PCP,CP0,PN,_T,_T,6.205536e+08,10721.00,0.000017,ok
4,AT,2022-H1,PCP,CW1,EUR,_T,_T,2.146843e+10,160542.30,0.000007,ok
5,AT,2022-H1,PCP,CW1,PN,_T,_T,9.642061e+07,470.00,0.000005,ok
6,AT,2022-H2,PCP,CP0,EUR,_T,_T,3.456834e+10,9115034.91,0.000264,ok
7,AT,2022-H2,PCP,CP0,EUR,_T,_T,2.880968e+10,1328025.81,0.000046,ok


Сохранено 1,020 строк × 47 полей; с целевой переменной 963.
Полный путь к CSV: C:\Users\txlort\Downloads\inputs\fraud_panel_2022_2024.csv


### Итоги 2.3.
#### *_Объединение источников._*

Шаг 1 — PCP + PCT. Вертикально (concat) в одну панель по общей схеме: total, fraud, fraud_rate, pair_status, source. Поля, которых нет у PCT (TYP_TRNSCTN, CRD_FNCTN и др.), остаются пустыми; источник различается по source, для PCT добавляется TYP_TRNSCTN = "CT". Уникальность — по DETAIL_KEY.

Шаг 2 — фильтр. Из 693 548 срезов остаются 1 020, у которых есть ряд F (иначе fraud_rate не считается).

Шаг 3 — Eurostat. Merge по ["REF_AREA", "year"]; перед этим EL → GR, отбор одной строки на «страна × год».

Шаг 4 — тексты EBA/ECB. Merge по ["year"]; годовой агрегат частот терминов.

Все merge — через checked_left: how="left", validate="many_to_one", проверка неизменности числа строк и ключей.

#### Почему CSV.
- Передача этапа: 2.3 заканчивается подготовкой, моделирование (2.4) читает готовую панель, не пересчитывая fraud_rate из сырых PCP/PCT.
- Воспроизводимость: не нужно снова парсить XML и JSON-stat (~3 млн строк сырья).
- Аудит: 47 колонок удобно смотреть вне ноутбука, сверять total/fraud/fraud_rate с ECB.
- Контракт колонок: фиксированный порядок — DETAIL_KEY, затем fraud_rate, pair_status, total, fraud, затем факторы.
- Проверка: round-trip чтение с assert на форму, ключи и fraud_rate.
- Журнал: сохраняются и 57 строк без целевой переменной — видно, какие срезы отброшены и почему (pair_status).

### 2.4. Подготовка выборок для моделей
В список включены семь полей. Среди строк с рассчитанной долей RMT_INTTN, INTTN_CHNNL, SCA и COUNT_AREA имеют по одному значению. У CRD_FNCTN среди заполненных строк тоже только одно значение, а в PCT поле отсутствует. Эти поля остаются в CSV для аудита, а в матрицу признаков не входят. Две текстовые частоты используются для интерпретации.

Для PN целевая переменная описывает долю по количеству операций, для EUR — по сумме в евро. Для них готовятся отдельные выборки. UNIT_MEASURE используется для разделения, а внутри каждой выборки остаются шесть признаков.

В CSV доля хранится в диапазоне от 0 до 1. Для моделей она умножается на 10 000. Это позволяет выражать ошибки MAE и RMSE в базисных пунктах. Сам по себе пересчёт единиц не меняет R², если одинаково пересчитать фактические и прогнозные значения. При последующей оценке модели данные следует делить по времени и проверять доступность внешних признаков на дату прогноза.

In [13]:
# Оставляем наблюдения с целевой долей и проверяем вариацию признаков.
MODEL_FEATURES = [
    "REF_AREA", "year", "half", "TYP_TRNSCTN", "UNIT_MEASURE",
    "RL_TRNSCTN", "online_purchase_share",
]
model_rows = panel.loc[panel["fraud_rate_available"]].copy()
model_X = model_rows[MODEL_FEATURES]
model_y = model_rows["fraud_rate"]
assert len(model_X) == len(model_y) and model_y.notna().all()
display(model_X.agg(["count", "nunique"]).T)

# Готовим две отдельные матрицы и целевые переменные в базисных пунктах.
features_by_unit = [field for field in MODEL_FEATURES if field != "UNIT_MEASURE"]
pn = model_rows["UNIT_MEASURE"].eq("PN")
eur = model_rows["UNIT_MEASURE"].eq("EUR")
model_X_pn, model_y_pn_bp = model_X.loc[pn, features_by_unit], model_y.loc[pn] * 10_000
model_X_eur, model_y_eur_bp = model_X.loc[eur, features_by_unit], model_y.loc[eur] * 10_000
assert (pn | eur).all() and not (pn & eur).any()
print(f"PN: {len(model_X_pn):,} строк. EUR: {len(model_X_eur):,} строк.")


,count,nunique
REF_AREA,963,24
year,963,3
half,963,2
TYP_TRNSCTN,963,3
UNIT_MEASURE,963,2
RL_TRNSCTN,963,2
online_purchase_share,963,68


PN: 528 строк. EUR: 435 строк.
